<a href="https://colab.research.google.com/github/ruhie18/customer-churn-revenue-intelligence/blob/main/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
url = "https://raw.githubusercontent.com/ruhie18/customer-churn-revenue-intelligence/refs/heads/main/data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print("Shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Shape (rows, columns): (7043, 21)

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


In [4]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [5]:

df['TotalCharges_numeric'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
problem_rows = df[df['TotalCharges_numeric'].isnull()]
print(f"Number of problem rows: {len(problem_rows)}")
problem_rows[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]

Number of problem rows: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


In [6]:

df['TotalCharges'] = df['TotalCharges_numeric']
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df = df.drop(columns=['TotalCharges_numeric'])
print("Missing values in TotalCharges now:", df['TotalCharges'].isnull().sum())
print("Data type of TotalCharges now:", df['TotalCharges'].dtype)

Missing values in TotalCharges now: 0
Data type of TotalCharges now: float64


In [7]:
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate customerIDs:", df['customerID'].duplicated().sum())

Fully duplicate rows: 0
Duplicate customerIDs: 0


In [8]:
for col in ['Contract', 'PaymentMethod', 'InternetService', 'Churn']:
    print(f"\n{col}:")
    print(df[col].value_counts())


Contract:
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

PaymentMethod:
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64

InternetService:
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

Churn:
Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [9]:
def tenure_group(tenure):
    if tenure <= 12:
        return '0-12 months'
    elif tenure <= 24:
        return '13-24 months'
    elif tenure <= 48:
        return '25-48 months'
    else:
        return '49+ months'

df['TenureGroup'] = df['tenure'].apply(tenure_group)

df['TenureGroup'].value_counts()

,count
TenureGroup,
49+ months,2239
0-12 months,2186
25-48 months,1594
13-24 months,1024


In [10]:
service_columns = ['PhoneService', 'MultipleLines', 'InternetService',
                    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                    'TechSupport', 'StreamingTV', 'StreamingMovies']

def count_services(row):
    count = 0
    for col in service_columns:
        if row[col] not in ['No', 'No internet service', 'No phone service']:
            count += 1
    return count

df['NumServices'] = df.apply(count_services, axis=1)

df['NumServices'].value_counts().sort_index()

,count
NumServices,
1,1264
2,859
3,846
4,965
5,922
6,908
7,676
8,395
9,208


In [11]:
df['MonthlyRevenueGroup'] = pd.cut(
    df['MonthlyCharges'],
    bins=[0, 35, 70, 100, df['MonthlyCharges'].max()],
    labels=['Low (<$35)', 'Medium ($35-70)', 'High ($70-100)', 'Very High ($100+)']
)

df['MonthlyRevenueGroup'].value_counts()

,count
MonthlyRevenueGroup,
High ($70-100),2681
Low (<$35),1735
Medium ($35-70),1725
Very High ($100+),902


In [12]:
def customer_value(row):
    if row['TotalCharges'] >= df['TotalCharges'].quantile(0.75):
        return 'High Value'
    elif row['TotalCharges'] >= df['TotalCharges'].quantile(0.25):
        return 'Medium Value'
    else:
        return 'Low Value'

df['CustomerValueGroup'] = df.apply(customer_value, axis=1)

df['CustomerValueGroup'].value_counts()

,count
CustomerValueGroup,
Medium Value,3522
High Value,1761
Low Value,1760


In [13]:
print("Final shape:", df.shape)
print("\nColumn list:")
print(df.columns.tolist())
print("\nFinal missing values check:")
print(df.isnull().sum().sum(), "total missing values")

df.head()

Final shape: (7043, 25)

Column list:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'TenureGroup', 'NumServices', 'MonthlyRevenueGroup', 'CustomerValueGroup']

Final missing values check:
0 total missing values


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,NumServices,MonthlyRevenueGroup,CustomerValueGroup
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-12 months,2,Low (<$35),Low Value
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,No,25-48 months,4,Medium ($35-70),Medium Value
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-12 months,4,Medium ($35-70),Low Value
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,25-48 months,4,Medium ($35-70),Medium Value
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-12 months,2,High ($70-100),Low Value


In [14]:
df.to_csv('cleaned_telco_churn.csv', index=False)
print("Saved cleaned_telco_churn.csv")

Saved cleaned_telco_churn.csv
